In [1]:
import json 
import pandas as pd 
import numpy as np 
import sys
from pathlib import Path
import re
import unicodedata

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from api.vietmap import UnifiedGeoCoder

In [2]:
BONBANH_DATASET_PATH=r"E:\car-price-prediction\data\bonbanh\used_cars.json"

In [3]:
df = pd.read_json(BONBANH_DATASET_PATH)

In [4]:
with open(r"E:\car-price-prediction\data\provinces.json", "rb") as f : 
    province = json.load(f)

normalize_province = {k.lower() : v.lower() for k, v in province.items()}
print(normalize_province)

{'hà nội': 'hà nội', 'thừa thiên huế': 'huế', 'lai châu': 'lai châu', 'quảng ninh': 'quảng ninh', 'thanh hóa': 'thanh hoá', 'nghệ an': 'nghệ an', 'điện biên': 'điện biên', 'sơn la': 'sơn la', 'lạng sơn': 'lạng sơn', 'hà tĩnh': 'hà tĩnh', 'cao bằng': 'cao bằng', 'tuyên quang': 'tuyên quang', 'hà giang': 'tuyên quang', 'lào cai': 'lào cai', 'yên bái': 'lào cai', 'thái nguyên': 'thái nguyên', 'bắc kạn': 'thái nguyên', 'phú thọ': 'phú thọ', 'vĩnh phúc': 'phú thọ', 'hòa bình': 'phú thọ', 'bắc ninh': 'bắc ninh', 'bắc giang': 'bắc ninh', 'hưng yên': 'hưng yên', 'thái bình': 'hưng yên', 'hải phòng': 'hải phòng', 'hải dương': 'hải phòng', 'ninh bình': 'ninh bình', 'nam định': 'ninh bình', 'hà nam': 'ninh bình', 'quảng bình': 'quảng trị', 'quảng trị': 'quảng trị', 'đà nẵng': 'đà nẵng', 'quảng nam': 'đà nẵng', 'quảng ngãi': 'quảng ngãi', 'kon tum': 'quảng ngãi', 'gia lai': 'gia lai', 'bình định': 'gia lai', 'khánh hòa': 'khánh hòa', 'ninh thuận': 'khánh hòa', 'lâm đồng': 'lâm đồng', 'bình thuận':

In [5]:
def find_province(x):
    def normalize_text(s):
        s = str(s).lower().strip()
        s = unicodedata.normalize("NFC", s)
        s = re.sub(r"\s+", " ", s)
        return s
    x = normalize_text(x)

    for id, pr in enumerate(normalize_province):
        if pr in x:
            return id

    return None

df["province"] = df["location"].apply(find_province)

In [6]:
df["province"]

0         0.0
1         0.0
2         0.0
3         0.0
4        45.0
         ... 
28472    45.0
28473    46.0
28474     8.0
28475     0.0
28476     4.0
Name: province, Length: 28477, dtype: float64

In [7]:
csv_path = r"E:\car-price-prediction\src\notebook\locations_geocoded.csv"
geo_df = pd.read_csv(csv_path)

In [8]:
geocoder = UnifiedGeoCoder()
geocoder.get_coordinates("Long Biên, đối diện Big C, Hà nội")

long biên, đối diện big c, hà nội
{'hà nội': (21.0283334, 105.854041), 'thừa thiên huế': (16.4639321, 107.5863388), 'lai châu': (22.2921668, 103.1798662), 'quảng ninh': (21.1718046, 107.2012742), 'thanh hóa': (19.9781573, 105.4816107), 'nghệ an': (19.1976001, 105.060676), 'điện biên': (21.6546566, 103.2168632), 'sơn la': (21.2276769, 104.1575944), 'lạng sơn': (21.8487579, 106.6140692), 'hà tĩnh': (18.3504832, 105.7623047), 'cao bằng': (22.7426936, 106.1060926), 'tuyên quang': (22.3382057, 105.0715846), 'hà giang': (22.8168007, 104.9504515), 'lào cai': (22.3069302, 104.1829592), 'yên bái': (21.8091042, 104.5182832), 'thái nguyên': (37.8699921, 112.5437075), 'bắc kạn': (22.2728552, 105.8564601), 'phú thọ': (21.3007538, 105.1349604), 'vĩnh phúc': (24.9835232, 109.9783989), 'hòa bình': (20.7090533, 105.2576439), 'bắc ninh': (21.3282166, 106.4625257), 'bắc giang': (21.3740092, 106.4663176), 'hưng yên': (20.6065846, 106.2843471), 'thái bình': (20.4031348, 106.5941615), 'hải phòng': (20.86232

(21.0283334, 105.854041)

In [9]:
def get_xy(location):
    lat, lon = geocoder.get_coordinates(str(location))
    return pd.Series({"lat": lat, "lon": lon})

mask = geo_df["lat"].isna()

geo_df.loc[mask, ["lat", "lon"]] = geo_df.loc[mask, "location"].apply(get_xy)

villa 05,06,07 lô c1 khu đô thị yên hòa, phường yên hòa, quận cầu giấy( cạnh công viên yên hoà ) hà nội
{'hà nội': (21.0283334, 105.854041), 'thừa thiên huế': (16.4639321, 107.5863388), 'lai châu': (22.2921668, 103.1798662), 'quảng ninh': (21.1718046, 107.2012742), 'thanh hóa': (19.9781573, 105.4816107), 'nghệ an': (19.1976001, 105.060676), 'điện biên': (21.6546566, 103.2168632), 'sơn la': (21.2276769, 104.1575944), 'lạng sơn': (21.8487579, 106.6140692), 'hà tĩnh': (18.3504832, 105.7623047), 'cao bằng': (22.7426936, 106.1060926), 'tuyên quang': (22.3382057, 105.0715846), 'hà giang': (22.8168007, 104.9504515), 'lào cai': (22.3069302, 104.1829592), 'yên bái': (21.8091042, 104.5182832), 'thái nguyên': (37.8699921, 112.5437075), 'bắc kạn': (22.2728552, 105.8564601), 'phú thọ': (21.3007538, 105.1349604), 'vĩnh phúc': (24.9835232, 109.9783989), 'hòa bình': (20.7090533, 105.2576439), 'bắc ninh': (21.3282166, 106.4625257), 'bắc giang': (21.3740092, 106.4663176), 'hưng yên': (20.6065846, 106.28

In [19]:
geo_df.to_csv(r"E:\car-price-prediction\src\notebook\locations_geocoded.csv")

In [12]:
merged_dataset = df.merge(
    geo_df,
    on="location",
    how="left"
)

In [13]:
merged_dataset= merged_dataset.drop("location", axis=1)

In [14]:
merged_dataset.head()

,url,brand,model,trim,date,price,year,status,origin,style,...,volume,exterior_color,interior_color,seats,doors,drive,odo,province,lat,lon
0,https://bonbanh.com/xe-audi-q6-e-tron-2026-631...,audi,q6,e-tron,2026-05-29,3000000000,2026,new,imported,suv,...,NaN,black,black,5,5,rfd,NaN,0.0,21.010445,105.788413
1,https://bonbanh.com/xe-audi-s6-sportback-e-tro...,audi,s6,sportback e-tron quattro,2026-05-28,4199000000,2026,new,imported,sedan,...,NaN,white,other,5,4,awd,NaN,0.0,21.010445,105.788413
2,https://bonbanh.com/xe-audi-q6-e-tron-2026-675...,audi,q6,e-tron,2026-05-30,3000000000,2026,new,imported,suv,...,NaN,silver,other,5,5,rfd,NaN,0.0,21.010445,105.788413
3,https://bonbanh.com/xe-bmw-5_series-520i-luxur...,bmw,5 series,520i luxury line,2026-05-30,1379000000,2022,used,imported,sedan,...,2.0,black,black,5,4,rfd,40000.0,0.0,21.052199,105.890768
4,https://bonbanh.com/xe-bmw-5_series--2015-6804865,bmw,5 series,520i,2026-05-16,590000000,2015,used,imported,sedan,...,2.0,black,cream,5,4,rfd,92000.0,45.0,10.784287,106.754756


In [16]:
merged_dataset.isna().sum()

url                  0
brand                0
model                0
trim                 0
date                 0
price                0
year                 0
status               0
origin               0
style                0
transmission         0
engine               0
volume            2078
exterior_color       0
interior_color       0
seats                0
doors                0
drive                0
odo               5584
province           503
lat                  0
lon                  0
dtype: int64

In [17]:
import os

out_path = r"E:\car-price-prediction\data\bonbanh\raw\used-car-prices.csv"

os.makedirs(os.path.dirname(out_path), exist_ok=True)

merged_dataset.to_csv(out_path, index=False, encoding="utf-8-sig")